# Entrenamiento

Smoke training temporal del Modelo 1 sin sobrescribir runtime.

In [ ]:
from pathlib import Path
import argparse
import sys
import tempfile
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'Suplematch-Backend').exists():
    ROOT = ROOT / 'Suplematch-Backend'
while ROOT.name != 'Suplematch-Backend' and ROOT.parent != ROOT:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from scripts.training import entrenar_modelo_condiciones as training
ROOT

## Insumos

In [ ]:
knowledge = training.load_knowledge()
training.validate_knowledge(knowledge)

## Smoke

In [ ]:
tmp_parent = ROOT / 'var'
tmp_parent.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix='suplematch_training_demo_', dir=tmp_parent) as tmp:
    tmp = Path(tmp)
    original_dirs = (training.TRAINING_DIR, training.REPORT_DIR, training.RUNTIME_DIR)
    training.TRAINING_DIR = tmp / 'training'
    training.REPORT_DIR = tmp / 'reports'
    training.RUNTIME_DIR = tmp / 'runtime'
    try:
        training.run_pipeline(argparse.Namespace(rows=500, seed=42))
        metrics_path = training.REPORT_DIR / '04_training_metrics.csv'
        model_path = training.RUNTIME_DIR / 'condition_mvp_model.pkl'
        metadata_path = training.RUNTIME_DIR / 'condition_mvp_metadata.csv'
        assert metrics_path.exists()
        assert model_path.exists()
        assert metadata_path.exists()
        display(pd.read_csv(metrics_path))
        result = {'model_bytes': model_path.stat().st_size, 'metadata_bytes': metadata_path.stat().st_size}
    finally:
        training.TRAINING_DIR, training.REPORT_DIR, training.RUNTIME_DIR = original_dirs
result

## Inferencia

In [ ]:
from app.ml.runtime.condition_mvp_inference import predict_condition_probabilities

features = {'edad': 34, 'sexo': 'masculino', 'tipo_dieta': 'omnivoro', 'dieta_deficiente': 1, 'fatiga_general': 1, 'meta_rendimiento': 1, 'fish_servings_week': 1, 'dairy_servings_day': 0, 'fruit_veg_servings_day': 2}
pd.DataFrame(predict_condition_probabilities(features)).head(10)